# Fixed gate bias with a balanced differential IV sweep

This example uses simulation only. The gate is a fixed external bias; Source 1 and Source 2 apply equal and opposite voltages so the differential target is `V1 - V2` while their common-mode voltage remains near zero.


## Audience and goals

This tutorial is for users who already run a single-source KohdaLab IV sweep. By the end, you will have connected a fixed gate and two sweep sources through one Rust engine session, applied a gate bias, run a balanced differential sweep, and verified gate cleanup.

1. Create a simulation-only output path.
2. Connect Source 1, Source 2, and the gate.
3. Apply a fixed gate voltage with explicit compliance.
4. Sweep the differential voltage and turn the gate off.


In [ ]:
from pathlib import Path
import tempfile
from kohdalab_iv_native import NativeIV

output = Path(tempfile.mkdtemp(prefix='iv-gate-balanced-')) / 'balanced.csv'


## Connect and run

The values below are intentionally simulation-only examples. A real device requires verified resources, wiring, and sample-approved limits before replacing `simulation=True`.


In [ ]:
with NativeIV() as iv:
    # One engine owns all three sources and the meter.
    iv.connect(simulation=True)
    iv.connect_source2(simulation=True, model='YOKOGAWA_GS210')
    iv.connect_gate(simulation=True, model='YOKOGAWA_GS210')

    # Gate compliance is an explicit hardware current limit.
    iv.set_gate_voltage(0.10, compliance=1e-3, ramp_step=1e-3)

    result = iv.run(output, {
        'mode': 'dc_iv', 'pattern': 'linear',
        'start': -1e-3, 'stop': 1e-3, 'step': 1e-3,
        'source_compliance': 1e-3,
        'balanced_source': {
            'source_model': 'YOKOGAWA_GS210',
            'source_compliance': 1e-3,
        },
        'settle_s': 0.0, 'start_settle_s': 0.0, 'ramp_wait_s': 0.0,
    })
    assert result['output'] == 'Off'
    assert result['gate_output'] == 'On'  # Gate is held during the IV run.
    iv.gate_output_off()
    assert iv.snapshot['gate_output'] == 'Off'

print(output)


For hardware, replace only the three `simulation=True` connections with verified resource strings and the actual models. Do not use another GUI, Notebook, or VISA session with these instruments at the same time. Both Source compliances and the gate compliance are physical safety limits; choose values for the connected device and sample before enabling output.


## Exercise

Repeat the simulation with a `0.05` V gate voltage and a new CSV filename. Confirm that the gate is still `On` after the IV run, then call `gate_output_off()`. A common mistake is creating separate clients for the gate and IV sweep; keep all devices in one `NativeIV` context so the engine owns the shared GPIB lease and shutdown sequence.


In [ ]:
# Answer scaffold (keep simulation=True):
# with NativeIV() as iv:
#     iv.connect(simulation=True)
#     iv.connect_source2(simulation=True)
#     iv.connect_gate(simulation=True)
#     iv.set_gate_voltage(0.05, compliance=1e-3, ramp_step=1e-3)
#     # run a balanced sweep to output.with_name('balanced-005V.csv')
#     iv.gate_output_off()
